In [3]:
# UnstructuredIO核心组件
from unstructured.partition.auto import partition
from typing import List
from unstructured.documents.elements import Element

# 使用partition函数自动检测文件类型并解析,默认strategy策略是auto，还会有fast策略，速度比image-to-text models的快100倍
elements: List[Element] = partition(filename="RAG评估.md", strategy="auto")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("==================")

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

print(type(elements[1]))
print(elements[4].text)

一.RAG效果评估的必要性
Title
{'category_depth': 0, 'languages': ['zho'], '_known_field_names': frozenset({'coordinates', 'is_extracted', 'enrichment_origins', 'parent_id', 'segment_start_seconds', 'page_name', 'chunk_index', 'last_modified', 'table_as_cells', 'emphasized_text_tags', 'filetype', 'sent_to', 'text_as_html', 'category_depth', 'header_footer_type', 'languages', 'is_continuation', 'table_id', 'subject', 'link_urls', 'routing_score', 'image_base64', 'routing', 'links', 'attached_to_filename', 'cc_recipient', 'link_texts', 'file_directory', 'page_number', 'image_path', 'url', 'detection_class_prob', 'image_url', 'image_mime_type', 'signature', 'table_extraction_method', 'data_source', 'detection_origin', 'num_carried_over_header_rows', 'segment_end_seconds', 'emphasized_text_contents', 'orig_elements', 'key_value_pairs', 'link_start_indexes', 'filename', 'bcc_recipient', 'email_message_id', 'sent_from'}), 'filename': 'RAG评估.md', 'filetype': 'text/markdown', 'last_modified': '2026-06-22

In [4]:
from unstructured.partition.auto import partition

# 🎯 只要加上这两个参数，让官方云端去帮你做高级版面分析
# 但是这个md太简单了，本地就能解析出来
elements = partition(
    filename="RAG评估.md",
    partition_via_api=True, # 走云端高精度模型
    api_key="RX1XicRJqOxKVUTCPAqIRyY8AmklRW"
)

for el in elements:
    print(f"[{el.category}] -> {el.text}")

[Title] -> 一.RAG效果评估的必要性
[Title] -> 二.RAG评估方法
[Title] -> 1.人工评估
[Title] -> 2.自动化评估
[Title] -> 3.LangSmith
[Title] -> 4.RAGAS
[Table] -> 维度 LangSmith RAGAS 核心定位 大模型应用的 集成开发平台 (调试、测试、评估、监控) 专门的RAG评估框架 ，用于量化RAG管道在不同组件层面上的性能 核心功能 提供全链路功能：应用 调试、测试、评估、监控 专注于评估 ，提供针对RAG的专用评估指标 评估方式 支持 自定义评估函数 和 基于参考答案的评估 (如精确匹配) ，以及 LLM即评委 等多种方式 提供一套 预设的、无需参考答案 的评估指标 ，可程序化计算 关键评估指标 支持广泛，取决于配置。可包括 精确匹配、工具调用准确性、自定义指标 等 忠实度、答案相关性、上下文精度、上下文召回率 等RAG核心指标 使用复杂度 相对较高，需要集成到开发流程中，配置数据集和评估器 相对较低，专注于评估，可通过几行代码对现有输入输出进行评估 数据需求 通常需要 构建包含输入和预期输出的测试数据集 无需参考答案 即可计算大部分核心指标
[Title] -> 三.评估指标


In [5]:
from typing import List, Dict, Any, Optional, Sequence
from pathlib import Path

# 自定义解析函数，支持任意类型的文件格式
def parse_file_with_unstructured(file_path: str):
    """
    使用UnstructuredIO解析单个文件

    Args:
        file_path: 文件路径

    Returns:
        Dict: 包含解析结果和统计信息的字典
    """
    print(f"\n 解析文件: {file_path}")

    try:
        # 使用partition函数自动检测文件类型并解析,默认strategy策略是auto，还会有fast策略，速度比image-to-text models的快100倍
        elements: List[Element] = partition(filename=file_path, strategy="auto")

        # 分析解析结果
        analysis = {
            "file_path": file_path,
            "file_extension": Path(file_path).suffix.lower(),
            "total_elements": len(elements),
            "element_types": {},
            "elements": elements,
            "text_content": "",
            "statistics": {}
        }

        # 统计元素类型
        for element in elements:
            element_type = type(element).__name__
            analysis["element_types"][element_type] = analysis["element_types"].get(element_type, 0) + 1

        # 提取文本内容
        text_parts = []

        for element in elements:
            if hasattr(element, 'text') and element.text:
                text_parts.append(element.text)

        analysis["text_content"] = "\n\n".join(text_parts)

        # 计算统计信息
        analysis["statistics"]["total_characters"] = len(analysis["text_content"])

        print(f"   解析完成")
        print(f"   元素总数: {analysis['total_elements']}")
        print(f"   元素类型: {analysis['element_types']}")
        print(f"   总字符数: {analysis['statistics']['total_characters']}")
        print(f"   文本内容: {analysis['text_content'][:200]} ")

    except Exception as e:
        print(f"文件解析失败: {e}")
        return {}

In [6]:
# %%markdown文档解析
parse_file_with_unstructured("RAG评估.md")


 解析文件: RAG评估.md
   解析完成
   元素总数: 8
   元素类型: {'Title': 7, 'Table': 1}
   总字符数: 474
   文本内容: 一.RAG效果评估的必要性

二.RAG评估方法

1.人工评估

2.自动化评估

3.LangSmith

4.RAGAS

维度 LangSmith RAGAS 核心定位 大模型应用的 集成开发平台 (调试、测试、评估、监控) 专门的RAG评估框架 ，用于量化RAG管道在不同组件层面上的性能 核心功能 提供全链路功能：应用 调试、测试、评估、监控 专注于评估 ，提供针对RAG的专用评估指标  


In [7]:
from unstructured.partition.md import partition_md
from typing import List
from unstructured.documents.elements import Element

# 使用partition_md函数检测markdown文件类型解析,include_page_breaks若希望在 Markdown 中标识页面断点（少见场景）
elements: List[Element] = partition_md(filename="RAG评估.md", languages=["zho"],include_page_breaks=True)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'category_depth': 0, 'languages': ['zho'], '_known_field_names': frozenset({'coordinates', 'is_extracted', 'enrichment_origins', 'parent_id', 'segment_start_seconds', 'page_name', 'chunk_index', 'last_modified', 'table_as_cells', 'emphasized_text_tags', 'filetype', 'sent_to', 'text_as_html', 'category_depth', 'header_footer_type', 'languages', 'is_continuation', 'table_id', 'subject', 'link_urls', 'routing_score', 'image_base64', 'routing', 'links', 'attached_to_filename', 'cc_recipient', 'link_texts', 'file_directory', 'page_number', 'image_path', 'url', 'detection_class_prob', 'image_url', 'image_mime_type', 'signature', 'table_extraction_method', 'data_source', 'detection_origin', 'num_carried_over_header_rows', 'segment_end_seconds', 'emphasized_text_contents', 'orig_elements', 'key_value_pairs', 'link_start_indexes', 'filename', 'bcc_recipient', 'email_message_id', 'sent_from'}), 'filename': 'RAG评估.md', 'filetype': 'text/markdown', 'last_modified': '2026-06-22T11:08:02'}
一.RAG效果评

In [8]:
# html文档解析
parse_file_with_unstructured("html-tags-decode.html")


 解析文件: html-tags-decode.html


   解析完成
   元素总数: 4
   元素类型: {'Title': 1, 'NarrativeText': 1, 'Text': 2}
   总字符数: 232
   文本内容: 识别和解析HTML标签

HTML tags (filter) decode, You can increase safety by filtering the danger label.

注：虽然此功能能极大地扩展 Markdown 语法，但也面临着安全上的风险，所以默认是不开启的。

Update: 可以通过设置 `settings.htmlDecode = "style,script,if 


In [9]:
from unstructured.partition.html import partition_html
from typing import List
from unstructured.documents.elements import Element

# 使用partition_html函数检测html网页类型解析
elements = partition_html(url="https://docs.unstructured.io/welcome",
                          headers={"User-Agent":"MyBot"},
                          ssl_verify=False,
                          include_page_breaks=False,
                          encoding="utf-8")
#elements: List[Element] = partition_html(url="https://docs.unstructured.io/welcome", languages=["zho"])

# 元素的元数据
print(elements[1].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[1].text)
print("===========================")

# 元素的类型
print(elements[1].category)
print("===========================")

/home/codespace/.local/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'docs.unstructured.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'link_texts': ['Learn about Unstructured’s product offerings'], 'link_urls': ['/about'], 'languages': ['eng'], '_known_field_names': frozenset({'coordinates', 'is_extracted', 'enrichment_origins', 'parent_id', 'segment_start_seconds', 'page_name', 'chunk_index', 'last_modified', 'table_as_cells', 'emphasized_text_tags', 'filetype', 'sent_to', 'text_as_html', 'category_depth', 'header_footer_type', 'languages', 'is_continuation', 'table_id', 'subject', 'link_urls', 'routing_score', 'image_base64', 'routing', 'links', 'attached_to_filename', 'cc_recipient', 'link_texts', 'file_directory', 'page_number', 'image_path', 'url', 'detection_class_prob', 'image_url', 'image_mime_type', 'signature', 'table_extraction_method', 'data_source', 'detection_origin', 'num_carried_over_header_rows', 'segment_end_seconds', 'emphasized_text_contents', 'orig_elements', 'key_value_pairs', 'link_start_indexes', 'filename', 'bcc_recipient', 'email_message_id', 'sent_from'}), 'filetype': 'text/html', 'url': '

In [10]:
parse_file_with_unstructured("销售数据统计.xlsx")


 解析文件: 销售数据统计.xlsx


   解析完成
   元素总数: 1
   元素类型: {'Table': 1}
   总字符数: 1940
   文本内容: 日期 销售人员ID 销量 销售金额 10/20/2018 3 100 300 10/14/2018 4 100 100 12/20/2018 5 400 1200 10/23/2018 2 300 900 11/16/2018 3 100 100 10/30/2018 5 400 800 11/5/2018 5 100 200 12/28/2018 1 100 300 11/20/2018 1 1 


In [11]:
from unstructured.partition.xlsx import partition_xlsx
from typing import List
from unstructured.documents.elements import Element

# 使用partition_xlsx函数检测excel文件类型并解析
elements: List[Element] = partition_xlsx(filename="销售数据统计.xlsx", languages=["zho"])

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text)
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'filename': '销售数据统计.xlsx', 'last_modified': '2026-06-22T11:08:02', 'page_name': 'Sheet1', 'page_number': 1, 'text_as_html': '<table><tr><td>日期</td><td>销售人员ID</td><td>销量</td><td>销售金额</td></tr><tr><td>10/20/2018</td><td>3</td><td>100</td><td>300</td></tr><tr><td>10/14/2018</td><td>4</td><td>100</td><td>100</td></tr><tr><td>12/20/2018</td><td>5</td><td>400</td><td>1200</td></tr><tr><td>10/23/2018</td><td>2</td><td>300</td><td>900</td></tr><tr><td>11/16/2018</td><td>3</td><td>100</td><td>100</td></tr><tr><td>10/30/2018</td><td>5</td><td>400</td><td>800</td></tr><tr><td>11/5/2018</td><td>5</td><td>100</td><td>200</td></tr><tr><td>12/28/2018</td><td>1</td><td>100</td><td>300</td></tr><tr><td>11/20/2018</td><td>1</td><td>100</td><td>200</td></tr><tr><td>10/16/2018</td><td>4</td><td>400</td><td>800</td></tr><tr><td>11/26/2018</td><td>2</td><td>200</td><td>400</td></tr><tr><td>10/25/2018</td><td>3</td><td>200</td><td>200</td></tr><tr><td>11/29/2018</td><td>1</td><td>400</td><td>800</td></tr><t

In [12]:
from unstructured.partition.csv import partition_csv
from typing import List
from unstructured.documents.elements import Element

# 使用partition_csv函数检测csv文件类型并解析
elements = partition_csv(filename="训练数据.csv", encoding="utf-8")

# 元素的元数据
#print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

months_as_customer age policy_number policy_bind_date policy_state policy_csl policy_deductable policy_annual_premium umbrella_limit insured_zip insured_sex insured_education_level insured_occupation insured_hobbies insured_relationship capital-gains capital-loss incident_date incident_type collision_type incident_severity authorities_contacted incident_state incident_city incident_location incide
Table


In [13]:
# word文档解析
parse_file_with_unstructured("数组.docx")


 解析文件: 数组.docx


   解析完成
   元素总数: 153
   元素类型: {'Title': 21, 'NarrativeText': 14, 'Text': 43, 'ListItem': 75}
   总字符数: 7656
   文本内容: 1. 数组简介 

1.1 概述

我们之前学习的变量或者是常量, 只能用来存储一个数据, 例如: 存储一个整数, 小数或者字符串等. 如果需要同时存储多个同类型的数据, 用变量或者常量来实现的话, 非常的繁琐. 针对于这种情况, 我们就可以通过数组来实现了.

例如: 假设某公司有50名员工, 现在需要统计该公司员工的工资情况, 例如计算平均工资、获取最高工资等。针对于这个需求，如果用前面所学的 


In [14]:
from unstructured.partition.docx import partition_docx
from unstructured.partition.doc import partition_doc
from typing import List
from unstructured.documents.elements import Element

# 使用partition_docx函数检测word文件类型并解析，include_page_breaks当文档支持 “分页” 时，以标识不同页的边界
elements = partition_docx(filename="数组.docx", encoding="utf-8", include_page_breaks=True)

# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[0].text[:400])
print("===========================")

# 元素的类型
print(elements[0].category)
print("===========================")

{'category_depth': 2, 'filename': '数组.docx', 'last_modified': '2026-06-22T11:08:02', 'languages': ['zho'], '_known_field_names': frozenset({'coordinates', 'is_extracted', 'enrichment_origins', 'parent_id', 'segment_start_seconds', 'page_name', 'chunk_index', 'last_modified', 'table_as_cells', 'emphasized_text_tags', 'filetype', 'sent_to', 'text_as_html', 'category_depth', 'header_footer_type', 'languages', 'is_continuation', 'table_id', 'subject', 'link_urls', 'routing_score', 'image_base64', 'routing', 'links', 'attached_to_filename', 'cc_recipient', 'link_texts', 'file_directory', 'page_number', 'image_path', 'url', 'detection_class_prob', 'image_url', 'image_mime_type', 'signature', 'table_extraction_method', 'data_source', 'detection_origin', 'num_carried_over_header_rows', 'segment_end_seconds', 'emphasized_text_contents', 'orig_elements', 'key_value_pairs', 'link_start_indexes', 'filename', 'bcc_recipient', 'email_message_id', 'sent_from'}), 'filetype': 'application/vnd.openxmlfo

### PDF解析

In [15]:
parse_file_with_unstructured("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf")


 解析文件: 甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf


2026-06-22 14:16:05.251030389 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+
No languages specified, defaulting to English.


   解析完成
   元素总数: 50
   元素类型: {'Title': 22, 'NarrativeText': 8, 'Header': 3, 'Text': 13, 'ListItem': 4}
   总字符数: 4273
   文本内容: 证 券 研 究 报 告

行 业 研 究

行 业 点 评

海外科技巨头持续发力 AI，龙头公司中报业绩亮眼

——AI 行业点评报告

◼ 核心观点 海外 AI 视角：（1）英伟达推出 B200A，2025 年 Blackwell GPU 有望 上量。《科创板日报》8 月 7 日讯，据 TrendForce 集邦咨询，英伟达仍 计划在 2024 年下半年推出 B100 及 B200，供应 CS 


In [19]:
from unstructured.partition.pdf import partition_pdf
from typing import List
from unstructured.documents.elements import Element
import io
import logging


print("🚀 启动【内存流绕道模式】，正在将 PDF 转化为二进制数据...")

# 1. 物理绕过：不在 partition_pdf 内部传 filename，而是我们自己用 Python 把文件读进内存
with open("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf", "rb") as f:
    pdf_bytes = f.read()

# 2. 将纯二进制数据包装成标准的内存文件流
file_like_object = io.BytesIO(pdf_bytes)

print("📡 正在跨过本地探查，直接向云端 API 发送纯净网络请求...")

# 使用partition_pdf函数检测pdf类型并解析

elements = partition_pdf(file=file_like_object,
                         partition_via_api=True,          # 🎯 核心：发给云端去解析，不吃本地 2G 内存！
                         api_key="RX1XicRJqOxKVUTCPAqIRyY8AmklRW34455",
                         strategy="hi_res", # 使用hi_res模式进行高精度解析
                         extract_images_in_pdf=False, # 提取pdf中的图片
                         #extract_image_block_types=["Table","Image"], # 提取表格和图片
                         #extract_image_block_output_dir="./images", # 保存图片到images目录
                         languages=["eng","zho"],
                         split_pdf_page=False, # 大文件分块处理，优化性能
                         infer_table_structure=False, # 是否尝试推断表格结构，会下载一个视觉目标检测的 Transformer 模型
                         include_page_breaks=True,
                         request_timeout=600) # 是否包含页码信息


# 元素的元数据
print(elements[0].metadata.__dict__)
print("===========================")

# 元素的文本内容
print(elements[1].text[:400])
print("===========================")

# 元素的类型
print(elements[1].category)
print("===========================")

🚀 启动【内存流绕道模式】，正在将 PDF 转化为二进制数据...
📡 正在跨过本地探查，直接向云端 API 发送纯净网络请求...


INFO: Reading PDF for file: /tmp/tmpg47jcq0s/document ...


{'detection_class_prob': 0.561924397945404, 'is_extracted': 'partial', 'coordinates': CoordinatesMetadata(points=((np.float64(24.472658157348633), np.float64(-4.65582799911499)), (np.float64(24.472658157348633), np.float64(1497.43310546875)), (np.float64(110.49360656738281), np.float64(1497.43310546875)), (np.float64(110.49360656738281), np.float64(-4.65582799911499))), system=<unstructured.documents.coordinates.PixelSpace object at 0x7b8ff8808590>), 'links': [], '_known_field_names': frozenset({'coordinates', 'is_extracted', 'enrichment_origins', 'parent_id', 'segment_start_seconds', 'page_name', 'chunk_index', 'last_modified', 'table_as_cells', 'emphasized_text_tags', 'filetype', 'sent_to', 'text_as_html', 'category_depth', 'header_footer_type', 'languages', 'is_continuation', 'table_id', 'subject', 'link_urls', 'routing_score', 'image_base64', 'routing', 'links', 'attached_to_filename', 'cc_recipient', 'link_texts', 'file_directory', 'page_number', 'image_path', 'url', 'detection_cl

In [20]:
from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError

# 1. 初始化云端客户端
s = UnstructuredClient(
    api_key_auth="RX1XicRJqOxKVUTCPAqIRyY8AmklRW" # 🧪 测试：改错这个 Key 就能看到 401 报错
)

filename = "甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf"

# 2. 读取文件
with open(filename, "rb") as f:
    file_content = f.read()

try:
    # 3. 🎯 核心修正：新版 SDK 直接在这里面通过命名参数传参，不需要构造 PartitionParameters 了
    res = s.general.partition(
        request={
            "partition_parameters": {
                "files": {
                    "content": file_content,
                    "file_name": filename,
                },
                "strategy": shared.Strategy.HI_RES, # 或者直接写 "hi_res"
                "languages": ["eng", "zho"],
            }
        }
    )
    
    # 4. 新版返回的对象直接就是 elements 列表，或者可以通过 res.elements 获取
    elements = res.elements if hasattr(res, "elements") else res
    
    print(f"成功！云端返回了 {len(elements)} 个元素。")
    if len(elements) > 0:
        # 打印前两个元素的内容和类型
        print(f"第一个元素类型: {type(elements[0])}")
        print(elements[:2])
    
except SDKError as e:
    print(f"抓到了！由于 Key 错误或网络问题，云端拒绝了请求：\n{e}")

INFO: split_pdf event=plan_created operation_id=3791a821-89bb-48f3-ba06-c960a248841e filename=甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf strategy=hi_res page_range=1-3 page_count=3 split_size=2 chunk_count=2 concurrency=5 allow_failed=False cache_mode=disabled timeout_seconds=None retry_config_mode=sdk_default_or_unset pool_max_connections=100 pool_max_keepalive=20 pool_keepalive_expiry=5.0s tls=trust_store=system-trust mtls_cert=none


INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_start operation_id=3791a821-89bb-48f3-ba06-c960a248841e chunk_count=2 concurrency=5 allow_failed=False client_timeout_seconds=None future_timeout_seconds=3605 num_waves=1
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: split_pdf event=batch_complete operation_id=3791a821-89bb-48f3-ba06-c960a248841e chunk_count=2 success_count=2 failure_count=0 transport_failure_count=0 elapsed_ms=30025 allow_failed=False


成功！云端返回了 49 个元素。
第一个元素类型: <class 'dict'>
[{'type': 'Header', 'element_id': '006f3cd235b449ec46792c57357f4c18', 'text': '证 券 研 究 报 告 行 业 研 究', 'metadata': {'is_extracted': 'partial', 'filetype': 'application/pdf', 'languages': ['eng', 'zho'], 'page_number': 1, 'filename': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf'}}, {'type': 'Header', 'element_id': '8155c45bd97c48c5b664171bddca17d8', 'text': '行 业 点 评', 'metadata': {'is_extracted': 'true', 'filetype': 'application/pdf', 'languages': ['eng', 'zho'], 'page_number': 1, 'filename': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf'}}]


In [13]:
from llama_index.core import SimpleDirectoryReader
from llama_parse import LlamaParse

# 如果文档结构复杂，优先使用 LlamaParse
# parser = LlamaParse(api_key="YOUR_LLAMA_CLOUD_API_KEY")
# documents = parser.load_data("sample.pdf")

# 或者使用简单读取器
documents = SimpleDirectoryReader(input_files=["RAG评估.md"]).load_data()

print(documents[0])
print("===========================")
print(documents[0].metadata)
print("===========================")
print(documents[0].text)
print("===========================")

ModuleNotFoundError: No module named 'llama_index'

In [24]:
from llama_index.readers.file.unstructured import UnstructuredReader
from pathlib import Path

reader = UnstructuredReader()
# 如果要控制是否启用 hi_res ，load_data中可以传参
documents = reader.load_data(file=Path("甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf"))

print("打印列表长度：" + str(len(documents)))
print("==================================")
print("打印解析的文本内容：" + documents[0].text[:100])
print("==================================")
print("打印元数据信息：" + str(documents[0].metadata))

2026-06-14 23:24:30,178 - WARNING - No languages specified, defaulting to English.
2026-06-14 23:24:30,198 - WARNING - 'doc_id' is deprecated and 'id_' will be used instead


打印列表长度：1
打印解析的文本内容：证 券 研 究 报 告

行 业 研 究

行 业 点 评

海外科技巨头持续发力 AI，龙头公司中报业绩亮眼

——AI 行业点评报告

◼ 核心观点 海外 AI 视角：（1）英伟达推出 B200A
打印元数据信息：{'coordinates': '{"points": [[6.84, 80.35964000000001], [6.84, 163.39963999999975], [20.88, 163.39963999999975], [20.88, 80.35964000000001]], "system": "PixelSpace", "layout_width": 595.32, "layout_height": 841.92}', 'filename': '甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf', 'last_modified': '2026-06-14T12:49:01', 'page_number': 1, 'languages': '["zho"]', 'filetype': 'application/pdf'}


In [ ]:
from unstructured.partition.auto import partition
# 使用LlamaIndex的Document对象，将解析后的元素转换为Document对象
from llama_index.core import Document

# 使用partition函数自动检测文件类型并解析
elements = partition(
    filename="甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
    strategy="hi_res",
    split_pdf_page=True,
    infer_table_structure=True,
    languages=["eng","chi_sim"])

# 将解析后的元素转换为Document对象
docs = [
    Document(text=e.text,
             metadata={"source":"甬兴证券-AI行业点评报告：海外科技巨头持续发力AI，龙头公司中报业绩亮眼.pdf",
                       "type": e.category})
    for e in elements]


In [26]:
from llama_index.core import Document
# 导入SentenceSplitter句子分割器
from llama_index.core.node_parser import SentenceSplitter

# 创建Document并设置元数据
doc = Document(
    text="这是一份关于RAG技术的文档...",
    metadata={
        "file_name": "rag_guide.pdf",
        "category": "技术文档",
        "author": "AI研究团队",
        "created_date": "2023-11-15"
    }
)

# 从Document创建Node时，元数据会自动传播
splitter = SentenceSplitter()
nodes = splitter.get_nodes_from_documents([doc])

# 每个node都会继承doc的metadata
nodes[0].metadata

{'file_name': 'rag_guide.pdf',
 'category': '技术文档',
 'author': 'AI研究团队',
 'created_date': '2023-11-15'}